In [ ]:
%pip install earthengine-api requests tqdm pandas numpy tensorflow geopandas torch --quiet

# Something went wrong
The HazardNet dashboard hit an unexpected error. Your saved data is safe — reload the app to continue monitoring hazards.

```
No QueryClient set, use QueryClientProvider to set one
```
Reload HazardNetGo to Dashboard

In [ ]:
import ee
import io
import json
import time
import urllib.request
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
import geopandas as gpd
from datetime import datetime, timedelta
import torch
import torch.nn.functional as F

# ==============================================================================
# 1. GEE AUTHENTICATION (Service Account)
# ==============================================================================
SERVICE_ACCOUNT = 'hazardnet-ee-service-kaggle@hazardnet-aas48424.iam.gserviceaccount.com'
CREDENTIALS_PATH = '/kaggle/input/datasets/ashifahmedshuvo/ee-token-json/hazardnet-aas48424-48d18edabfcc.json'

try:
    credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, CREDENTIALS_PATH)
    ee.Initialize(credentials)  
    print("OK: Google Earth Engine Initialized via Service Account")
except Exception as e:
    print(f"Warning: GEE Initialization Failed: {e}")
    print("   Please verify the service account path and permissions in Kaggle Secrets/Dataset.")

# ==============================================================================
# 2. CONFIGURATION & PATHS
# ==============================================================================
BAND_NAMES = [
    'SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR', 
    'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp', 
    'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad'
]

MODEL_PATH = '/kaggle/input/notebooks/ashifahmedshuvo/hazardnet-model-conversion/HazardNet_Deployment_Bundles/deployment_bundle/hazardnet_fp32.tflite'
STATS_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet-datasets/tensors_output/normalization_stats.json'
OUTPUT_CSV = '/kaggle/working/hazardnet_forecasts_latest.csv'

HORIZONS = {'7_days': 7, '15_days': 15}
HAZARD_CLASSES = ['Cold Wave', 'Drought', 'Fire', 'Flash Flood', 
                  'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone']

# ==============================================================================
# 3. LOAD BANGLADESH FAO GAUL ADMINISTRATIVE BOUNDARIES (Native GEE)
# ==============================================================================
def load_fao_gaul_boundaries():
    print("Loading FAO GAUL Administrative Boundaries for Bangladesh via GEE...")
    
    bd_filter = ee.Filter.eq('ADM0_NAME', 'Bangladesh')
    gaul_adm0 = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(bd_filter)
    gaul_adm1 = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(bd_filter)
    gaul_adm2 = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(bd_filter)
    
    print(f"   ADM0 (Country): {gaul_adm0.size().getInfo()} feature(s)")
    print(f"   ADM1 (Divisions): {gaul_adm1.size().getInfo()} features")
    print(f"   ADM2 (Districts): {gaul_adm2.size().getInfo()} features")
    
    def extract_props(feat):
        geom = feat.geometry()
        centroid = geom.centroid()
        return feat.set({
            'lon': centroid.coordinates().get(0),
            'lat': centroid.coordinates().get(1),
            'ADM2_NAME': feat.get('ADM2_NAME'),
            'ADM1_NAME': feat.get('ADM1_NAME'),
            'ADM2_PCODE': feat.get('ADM2_PCODE')
        })
    
    gaul_adm2_mapped = gaul_adm2.map(extract_props)
    features = gaul_adm2_mapped.getInfo()['features']
    
    districts = []
    sorted_features = sorted(features, key=lambda x: x['properties'].get('ADM2_NAME', ''))
    
    for i, feat in enumerate(sorted_features):
        props = feat['properties']
        districts.append({
            "id": i + 1,
            "name": str(props.get('ADM2_NAME', 'Unknown')).strip(),
            "division": str(props.get('ADM1_NAME', 'Unknown')).strip(),
            "pcode": str(props.get('ADM2_PCODE', props.get('ADM2_CODE', ''))).strip(),
            "lat": round(float(props.get('lat', 0)), 4),
            "lon": round(float(props.get('lon', 0)), 4)
        })
        
    divisions = set(d['division'] for d in districts)
    print(f"\nOK: Loaded {len(districts)} districts across {len(divisions)} divisions via FAO GAUL")
    
    return None, None, None, None, districts

adm0, adm1, adm2, capitals, DISTRICTS = load_fao_gaul_boundaries()

print(f"\nFirst 5 districts:")
for d in DISTRICTS[:5]:
    print(f"   {d['id']:2d}. {d['name']:20s} | {d['division']:12s} | ({d['lat']}, {d['lon']}) | {d['pcode']}")

# ==============================================================================
# 4. ROBUST GEE PIPELINE
# ==============================================================================
def harmonize_and_rename(image, mission_type):
    try:
        mappings = {
            'L57': {'src': ['SR_B1', 'SR_B3', 'SR_B4', 'SR_B5'], 'dest': ['Blue', 'Red', 'NIR', 'SWIR']},
            'L8':  {'src': ['SR_B2', 'SR_B4', 'SR_B5', 'SR_B6'], 'dest': ['Blue', 'Red', 'NIR', 'SWIR']},
            'S2':  {'src': ['B2', 'B4', 'B8', 'B11'],           'dest': ['Blue', 'Red', 'NIR', 'SWIR']}
        }
        selected_map = mappings.get(mission_type)
        band_names = image.bandNames()
        count = band_names.size()
        
        return ee.Image(ee.Algorithms.If(
            count.gte(4),
            image.select(selected_map['src']).rename(selected_map['dest']),
            ee.Image.constant([0, 0, 0, 0]).rename(selected_map['dest']).updateMask(0)
        ))
    except Exception as e:
        return None

def get_hybrid_optical(region, start, end):
    s2_col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(region).filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    if s2_col.size().getInfo() > 0:
        return harmonize_and_rename(s2_col.median(), 'S2').unmask(0)

    l8_col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
        .filterBounds(region).filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
    if l8_col.size().getInfo() > 0:
        return harmonize_and_rename(l8_col.median(), 'L8').resample('bicubic').unmask(0)

    l7_col = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
        .filterBounds(region).filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
    if l7_col.size().getInfo() > 0:
        return harmonize_and_rename(l7_col.median(), 'L57').resample('bicubic').unmask(0)

    return ee.Image.constant([0, 0, 0, 0]).rename(['Blue', 'Red', 'NIR', 'SWIR']).float().unmask(0)

def get_temporal_15ch_stack(region, start, end):
    try:
        S1_BANDS = ['VV', 'VH']
        ERA5_BANDS = ['temperature_2m', 'total_precipitation_sum', 'temperature_2m_max',
                      'temperature_2m_min', 'volumetric_soil_water_layer_1',
                      'volumetric_soil_water_layer_3', 'soil_temperature_level_1',
                      'dewpoint_temperature_2m', 'surface_solar_radiation_downwards_sum']

        s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterBounds(region).filterDate(start, end) \
            .filter(ee.Filter.eq('instrumentMode', 'IW')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        
        default_s1 = ee.Image.constant([0, 0]).rename(S1_BANDS).float()
        s1 = ee.Image(ee.Algorithms.If(
            s1_collection.size().gt(0),
            s1_collection.select(S1_BANDS).median().unmask(0),
            default_s1
        ))

        s2_hybrid = get_hybrid_optical(region, start, end)

        era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
            .filterBounds(region).filterDate(start, end) \
            .median().resample('bilinear').unmask(0).select(ERA5_BANDS)

        return s1.addBands(s2_hybrid).addBands(era5).float().clip(region).unmask(0)
    except Exception as e:
        print(f"Stack Construction Error: {e}")
        return None

# ==============================================================================
# 5. NUMPY DOWNLOAD & TENSOR BUILDER
# ==============================================================================
def get_ee_image_as_numpy(image, region, scale=10, target_size=(64, 64)):
    url = image.getDownloadURL({'region': region, 'scale': scale, 'format': 'NPY'})
    response = urllib.request.urlopen(url)
    data = np.load(io.BytesIO(response.read()), allow_pickle=True)
    
    bands = []
    for b in image.bandNames().getInfo():
        bands.append(data[b])
    
    img_np = np.stack(bands, axis=0)
    h, w = img_np.shape[1], img_np.shape[2]
    
    if h != target_size[0] or w != target_size[1]:
        tensor = torch.from_numpy(img_np).float().unsqueeze(0)
        tensor_resized = F.interpolate(tensor, size=target_size, mode='bilinear', align_corners=False)
        img_np = tensor_resized.squeeze(0).numpy()
        
    return img_np

def get_openmeteo_forecast(lat, lon, horizon_days):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min,"
                 "precipitation_sum,dew_point_2m_mean,shortwave_radiation_sum,"
                 "wind_speed_10m_max,et0_fao_evapotranspiration_sum",
        "timezone": "Asia/Dhaka",
        "forecast_days": min(horizon_days + 1, 16)
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        daily = data.get('daily', {})
        
        def safe_array(key, default_val):
            arr = daily.get(key)
            if arr is None:
                arr = [default_val]
            arr = [float(x) if x is not None else np.nan for x in arr]
            return np.array(arr)

        temp_mean = safe_array('temperature_2m_mean', 295.0)
        temp_max = safe_array('temperature_2m_max', 300.0)
        temp_min = safe_array('temperature_2m_min', 280.0)
        precip = safe_array('precipitation_sum', 0.0)
        dewpoint = safe_array('dew_point_2m_mean', 285.0)
        solar_rad = safe_array('shortwave_radiation_sum', 5000.0)
        wind_max = safe_array('wind_speed_10m_max', 0.0)
        et_sum = safe_array('et0_fao_evapotranspiration_sum', 0.0)
        
        return {
            'Temp_2m': float(np.nanmean(temp_mean)),
            'Precip': float(np.nansum(precip)) / 1000.0,
            'Max_Temp': float(np.nanmax(temp_max)),
            'Min_Temp': float(np.nanmin(temp_min)),
            'Dewpoint': float(np.nanmean(dewpoint)),
            'Solar_Rad': float(np.nansum(solar_rad)) * 1000.0,
            'Wind_Max': float(np.nanmax(wind_max)),
            'ET_Sum': float(np.nansum(et_sum))
        }
    except requests.exceptions.RequestException as e:
        print(f"Open-Meteo HTTP Error for ({lat}, {lon}): {e}")
        return None
    except Exception as e:
        print(f"Open-Meteo Processing Error for ({lat}, {lon}): {e}")
        return None

def build_future_tensor(lat, lon, horizon_days, norm_stats):
    today = datetime.now()
    region = ee.Geometry.Point([lon, lat]).buffer(320).bounds().getInfo()
    
    om_data = get_openmeteo_forecast(lat, lon, horizon_days)
    if not om_data: 
        print("Open-Meteo forecast failed.")
        return None, None
    
    historical_steps = []
    
    for t in range(9, 0, -1): 
        end_date = today - timedelta(days=(t-1)*10)
        start_date = end_date - timedelta(days=10)
        
        combined_img = get_temporal_15ch_stack(region, start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        
        if combined_img:
            try:
                step_np = get_ee_image_as_numpy(combined_img, region, scale=10)
                historical_steps.append(step_np)
            except Exception as e:
                print(f"GEE Download Error at T-{t}: {e}")
                return None, None
        else:
            print(f"GEE Stack Construction Failed at T-{t}")
            return None, None

    t0_start = (today - timedelta(days=10)).strftime('%Y-%m-%d')
    t0_end = today.strftime('%Y-%m-%d')
    
    t0_combined_img = get_temporal_15ch_stack(region, t0_start, t0_end)
    if t0_combined_img:
        try:
            t0_np = get_ee_image_as_numpy(t0_combined_img, region, scale=10)
            
            om_bands = [
                om_data['Temp_2m'], om_data['Precip'], om_data['Max_Temp'], om_data['Min_Temp'],
                0.3, 0.3, 290.0, om_data['Dewpoint'], om_data['Solar_Rad']
            ]
            t0_np[6:15, :, :] = np.array(om_bands).reshape(9, 1, 1)
            historical_steps.append(t0_np)
        except Exception as e:
            print(f"T-0 Construction Error: {e}")
            return None, None
    else:
        print("T-0 Stack Construction Failed")
        return None, None

    full_tensor = np.stack(historical_steps, axis=0)
    normalized = np.zeros_like(full_tensor, dtype=np.float32)
    
    for c, band in enumerate(BAND_NAMES):
        mean = norm_stats[band]['mean']
        std = max(norm_stats[band]['std'], 1e-6)
        normalized[:, c, :, :] = (full_tensor[:, c, :, :] - mean) / std
        
    tflite_input = np.transpose(normalized, (0, 2, 3, 1))
    return np.expand_dims(tflite_input, axis=0).astype(np.float32), om_data

# ==============================================================================
# 6. HYBRID COGNITIVE: PHYSICAL INDEX FORMULAS (Integrated)
# ==============================================================================
def safe_float(val, default=0.0):
    try:
        f = float(val)
        return default if np.isnan(f) else f
    except Exception:
        return default

def om_calc_severe_storm(precip_max, wind_max):
    p = safe_float(precip_max, 0.0) / 100.0
    w = max(safe_float(wind_max, 0.0) - 50.0, 0.0) / 100.0
    return float(np.clip(0.6 * w + 0.4 * min(p, 1.0), 0.0, 1.0))

def om_calc_cold_wave(temp_min_celsius, duration_days):
    cold_anomaly = np.clip((16.0 - safe_float(temp_min_celsius, 16.0)) / 10.0, 0.0, 1.0)
    duration_factor = np.clip(safe_float(duration_days, 1.0) / 5.0, 0.0, 1.0)
    return float(np.clip(0.7 * cold_anomaly + 0.3 * duration_factor, 0.0, 1.0))

def om_calc_fire(temp_max_celsius, wind_max, et_sum):
    heat = np.clip((safe_float(temp_max_celsius, 30.0) - 25.0) / 15.0, 0.0, 1.0)
    wind = np.clip((safe_float(wind_max, 10.0) - 5.0) / 20.0, 0.0, 1.0)
    dryness = np.clip(safe_float(et_sum, 3.0) / 6.0, 0.0, 1.0)
    return float(np.clip(0.4 * heat + 0.3 * wind + 0.3 * dryness, 0.0, 1.0))

def om_calc_tropical_cyclone(wind_max, precip_sum_mm):
    w = max(safe_float(wind_max, 0.0) - 50.0, 0.0) / 150.0
    p = safe_float(precip_sum_mm, 0.0) / 300.0
    return float(np.clip(0.7 * min(w, 1.0) + 0.3 * min(p, 1.0), 0.0, 1.0))

def om_calc_drought(temp_max_celsius, precip_sum_mm):
    temp_stress = np.clip((safe_float(temp_max_celsius, 25.0) - 25.0) / 20.0, 0.0, 1.0)
    precip_deficit = np.clip((200.0 - safe_float(precip_sum_mm, 200.0)) / 200.0, 0.0, 1.0)
    return float(np.clip(0.6 * temp_stress + 0.4 * precip_deficit, 0.0, 1.0))

def om_calc_flood(precip_sum_mm, precip_max_mm):
    p_factor = np.clip(safe_float(precip_sum_mm, 0.0) / 300.0, 0.0, 1.0)
    i_factor = np.clip(safe_float(precip_max_mm, 0.0) / 100.0, 0.0, 1.0)
    return float(np.clip(0.5 * p_factor + 0.5 * i_factor, 0.0, 1.0))

def om_calc_heat_wave(temp_max_celsius, duration_days):
    temp_anomaly = np.clip((safe_float(temp_max_celsius, 30.0) - 30.0) / 15.0, 0.0, 1.0)
    duration_factor = np.clip(safe_float(duration_days, 1.0) / 5.0, 0.0, 1.0)
    return float(np.clip(0.7 * temp_anomaly + 0.3 * duration_factor, 0.0, 1.0))

# ==============================================================================
# 7. MODEL INFERENCE SETUP
# ==============================================================================
print("\nLoading TFLite Model and Normalization Stats...")
with open(STATS_PATH, 'r') as f:
    NORM_STATS = json.load(f)

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def run_inference(tensor):
    interpreter.set_tensor(input_details[0]['index'], tensor)
    interpreter.invoke()
    
    hazard_logits = interpreter.get_tensor(output_details[0]['index'])[0]
    severity_score = interpreter.get_tensor(output_details[1]['index'])[0]
    
    probs = tf.nn.softmax(hazard_logits).numpy()
    pred_class = np.argmax(probs)
    confidence = float(probs[pred_class])
    
    return HAZARD_CLASSES[pred_class], confidence, float(severity_score)

# ==============================================================================
# 8. OPTIMIZED MAIN EXECUTION LOOP
# ==============================================================================
def fetch_historical_steps(lat, lon, norm_stats):
    today = datetime.now()
    region = ee.Geometry.Point([lon, lat]).buffer(320).bounds().getInfo()
    historical_steps = []
    
    for t in range(9, 0, -1): 
        end_date = today - timedelta(days=(t-1)*10)
        start_date = end_date - timedelta(days=10)
        
        combined_img = get_temporal_15ch_stack(region, start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
        
        if combined_img:
            try:
                step_np = get_ee_image_as_numpy(combined_img, region, scale=10)
                historical_steps.append(step_np)
            except Exception as e:
                print(f"GEE Download Error at T-{t} for {lat},{lon}: {e}")
                return None
        else:
            print(f"GEE Stack Construction Failed at T-{t} for {lat},{lon}")
            return None
            
    return historical_steps

def build_t0_and_infer(dist, historical_steps, horizon_days, norm_stats):
    lat, lon = dist['lat'], dist['lon']
    today = datetime.now()
    region = ee.Geometry.Point([lon, lat]).buffer(320).bounds().getInfo()
    
    om_data = get_openmeteo_forecast(lat, lon, horizon_days)
    if not om_data: 
        return None, None
        
    t0_start = (today - timedelta(days=10)).strftime('%Y-%m-%d')
    t0_end = today.strftime('%Y-%m-%d')
    
    t0_combined_img = get_temporal_15ch_stack(region, t0_start, t0_end)
    if t0_combined_img:
        try:
            t0_np = get_ee_image_as_numpy(t0_combined_img, region, scale=10)
            
            om_bands = [
                om_data['Temp_2m'], om_data['Precip'], om_data['Max_Temp'], om_data['Min_Temp'],
                0.3, 0.3, 290.0, om_data['Dewpoint'], om_data['Solar_Rad']
            ]
            t0_np[6:15, :, :] = np.array(om_bands).reshape(9, 1, 1)
            
            full_tensor = np.stack(historical_steps + [t0_np], axis=0)
            normalized = np.zeros_like(full_tensor, dtype=np.float32)
            
            for c, band in enumerate(BAND_NAMES):
                mean = norm_stats[band]['mean']
                std = max(norm_stats[band]['std'], 1e-6)
                normalized[:, c, :, :] = (full_tensor[:, c, :, :] - mean) / std
                
            tflite_input = np.transpose(normalized, (0, 2, 3, 1))
            tflite_input = np.expand_dims(tflite_input, axis=0).astype(np.float32)
            
            return tflite_input, om_data
        except Exception as e:
            print(f"T-0 Construction Error for {dist['name']}: {e}")
            return None, None
    return None, None

# --- Execute Optimized Loop ---
results = []
print(f"\nStarting Optimized HazardNet Forecast Pipeline for {len(DISTRICTS)} Districts...")
print("Fetching historical data once per district, then projecting 2 horizons.\n")

for dist in DISTRICTS:
    print(f"Processing {dist['name']} ({dist['id']}/{len(DISTRICTS)})...")
    
    historical_steps = fetch_historical_steps(dist['lat'], dist['lon'], NORM_STATS)
    
    if not historical_steps:
        print(f"Skipped {dist['name']} due to historical data failure.")
        continue
        
    for horizon_name, days in HORIZONS.items():
        target_date = (datetime.now() + timedelta(days=days)).strftime('%Y-%m-%d')
        
        tensor, om_data = build_t0_and_infer(dist, historical_steps, days, NORM_STATS)
        
        if tensor is not None and om_data is not None:
            hazard, conf, severity = run_inference(tensor)
            
            temp_max_c = om_data['Max_Temp'] - 273.15
            temp_min_c = om_data['Min_Temp'] - 273.15
            precip_mm = om_data['Precip'] * 1000.0
            
            physics_severity = 0.50
            if hazard == 'Tropical Cyclone':
                physics_severity = om_calc_tropical_cyclone(om_data['Wind_Max'], precip_mm)
            elif hazard == 'Severe Local Storm':
                physics_severity = om_calc_severe_storm(precip_mm, om_data['Wind_Max'])
            elif hazard == 'Cold Wave':
                physics_severity = om_calc_cold_wave(temp_min_c, days)
            elif hazard == 'Fire':
                physics_severity = om_calc_fire(temp_max_c, om_data['Wind_Max'], om_data['ET_Sum'])
            elif hazard == 'Drought':
                physics_severity = om_calc_drought(temp_max_c, precip_mm)
            elif hazard in ['Flood', 'Flash Flood']:
                physics_severity = om_calc_flood(precip_mm, precip_mm)
            elif hazard == 'Heat Wave':
                physics_severity = om_calc_heat_wave(temp_max_c, days)
            
            # REFACTORED: Appended Open-Meteo forecast variables district-wise
            results.append({
                'district_id': dist['id'],
                'district_name': dist['name'],
                'division': dist['division'],
                'pcode': dist['pcode'],
                'horizon': horizon_name,
                'hazard_type': hazard,
                'model_severity': round(severity, 4),
                'physics_severity': round(physics_severity, 4),
                'confidence': round(conf, 4),
                'target_date': target_date,
                'prediction_date': datetime.now().strftime('%Y-%m-%d'),
                'data_source': 'Hybrid_Cognitive_Forecast',
                # Open-Meteo Forecast Variables (District-wise)
                'om_temp_2m_k': round(om_data['Temp_2m'], 4),
                'om_precip_m': round(om_data['Precip'], 4),
                'om_max_temp_k': round(om_data['Max_Temp'], 4),
                'om_min_temp_k': round(om_data['Min_Temp'], 4),
                'om_dewpoint_k': round(om_data['Dewpoint'], 4),
                'om_solar_rad_j': round(om_data['Solar_Rad'], 4),
                'om_wind_max_ms': round(om_data['Wind_Max'], 4),
                'om_et_sum_m': round(om_data['ET_Sum'], 4)
            })
        else:
            print(f"Failed to process {dist['name']} ({horizon_name})")
            
        time.sleep(0.2)

# ==============================================================================
# 9. SAVE & VERIFY OUTPUT
# ==============================================================================
df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False)

print("\n" + "="*60)
print("FORECAST PIPELINE COMPLETE")
print("="*60)
print(f"Total Predictions: {len(df_results)}")
print(f"Output Saved To: {OUTPUT_CSV}")
print("\nSample Output:")
print(df_results.head(10).to_string())